# Audit Tracer: laboratorio de validación

Este notebook acompaña la **Guía de pruebas de validación**. Cada sección lleva el código del escenario de la guía (A1, B3, C4...).

**Cómo usarlo**
1. Completa la celda de configuración y ejecuta las celdas **en orden**.
2. Cada prueba imprime `CUMPLE` o `NO CUMPLE`, con lo esperado y lo obtenido. Lee el código: puedes cambiar los ataques o agregar los tuyos.
3. La celda final resume tus resultados.

Los datos son sintéticos. Las pruebas de manipulación de archivos solo afectan a este entorno de Colab.

In [ ]:
# ── CONFIGURACIÓN ──────────────────────────────────────────────
API_URL = "https://audit-tracer-deploy.onrender.com"   # servidor central
EVALUADOR = "p1"                                       # p1 o p2, según tu hoja de acceso
INSTALAR = True
PAQUETE = "git+https://github.com/Alexis1661/audit-tracer-deploy.git"

In [ ]:
import subprocess, sys
if INSTALAR:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", PAQUETE, "flask"])
print("Listo.")

In [ ]:
# Utilidades del laboratorio. No importan pandas ni audit_tracer.
import base64, csv, datetime as dt, getpass, json, os, pathlib, sqlite3, tempfile, time, uuid, zlib
import requests

API_URL = API_URL.rstrip("/")
TIMEOUT = 120  # un servidor gratuito dormido tarda cerca de un minuto en responder
RESULTADOS = []
_CONTRASENAS = globals().get("_CONTRASENAS", {})
TRABAJO = pathlib.Path(tempfile.mkdtemp(prefix="validacion_"))
os.chdir(TRABAJO)


def check(escenario, descripcion, cumple, obtenido=""):
    RESULTADOS.append({"escenario": escenario, "prueba": descripcion, "cumple": bool(cumple), "obtenido": str(obtenido)})
    print(f"[{'CUMPLE' if cumple else 'NO CUMPLE'}] {escenario} · {descripcion}" + (f"  →  {obtenido}" if obtenido != "" else ""))


def correo(cuenta):
    return f"{cuenta}.{EVALUADOR}@validacion.audit"


def contrasena(cuenta):
    if cuenta not in _CONTRASENAS:
        _CONTRASENAS[cuenta] = getpass.getpass(f"Contraseña de {correo(cuenta)}: ")
    return _CONTRASENAS[cuenta]


def contenido_cookie(cookie):
    """Lee la cookie de sesión de Flask: va firmada, no cifrada."""
    comprimida = cookie.startswith(".")
    cuerpo = cookie.lstrip(".").split(".")[0]
    datos = base64.urlsafe_b64decode(cuerpo + "=" * (-len(cuerpo) % 4))
    return json.loads(zlib.decompress(datos) if comprimida else datos)


def sesion_web(cuenta):
    s = requests.Session()
    r = s.post(f"{API_URL}/login", data={"email": correo(cuenta), "password": contrasena(cuenta)},
               allow_redirects=False, timeout=TIMEOUT)
    if r.status_code != 302:
        raise RuntimeError(f"No se pudo iniciar sesión como {cuenta}: HTTP {r.status_code}")
    s.usuario_id = contenido_cookie(s.cookies["session"])["usuario_id"]
    return s


def sincronizar(token, evento):
    cabeceras = {"Authorization": f"Bearer {token}"} if token else {}
    return requests.post(f"{API_URL}/api/eventos/sincronizar", json=evento, headers=cabeceras, timeout=TIMEOUT)


def filas_eventos(sesion, **filtros):
    """Cuenta las filas de la primera página de /eventos (máximo 10)."""
    return sesion.get(f"{API_URL}/eventos", params=filtros, timeout=TIMEOUT).text.count('class="event-id"')


print("Despertando el servidor...")
print("Servidor:", requests.get(f"{API_URL}/login", timeout=TIMEOUT).status_code)

---
# Bloque A · Captura transparente y atribución

## A1 · Actividad sin identificar
**Propiedad:** todo evento queda registrado, aunque no se sepa quién lo generó.
**Esperado:** la carga queda como `DESCONOCIDO` y `CRITICO`.

In [ ]:
import audit_tracer
import pandas as pd
from audit_tracer.utils.paths import local_db_path
import random

LOCAL = local_db_path()
random.seed(7)
with open("pacientes_sinteticos.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id_paciente", "edad", "sexo", "glucosa", "presion", "codigo_dx", "notas"])
    for i in range(2500):
        w.writerow([f"P{i:05d}", random.randint(1, 95), random.choice(["F", "M", ""]),
                    random.choice(["", random.randint(70, 180)]), random.randint(90, 170),
                    random.choice(["E11", "I10", "J45"]), "texto clínico sintético"])
with open("diagnosticos.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerows([["codigo_dx", "descripcion"], ["E11", "Diabetes tipo 2"], ["I10", "Hipertensión"], ["J45", "Asma"]])


def ultimos_eventos(n=1, **igual):
    conn = sqlite3.connect(LOCAL)
    conn.row_factory = sqlite3.Row
    condicion = " AND ".join(f"{k} = ?" for k in igual) or "1=1"
    filas = conn.execute(f"SELECT * FROM audit_log WHERE {condicion} ORDER BY event_id DESC LIMIT ?",
                         (*igual.values(), n)).fetchall()
    conn.close()
    return [dict(f) for f in filas]


_ = pd.read_csv("pacientes_sinteticos.csv")
e = ultimos_eventos()[0]
check("A1", "Carga sin identificar registrada como DESCONOCIDO y CRITICO",
      e["tipo_accion"] == "CARGA" and e["usuario_id"] == "DESCONOCIDO" and e["nivel_alerta"] == "CRITICO",
      f"{e['tipo_accion']} · {e['usuario_id']} · {e['nivel_alerta']}")

## A2 · Login por código de dispositivo y atribución
**Propiedad:** autenticación sin escribir contraseñas en el notebook.
**Qué hacer:** ejecuta la celda, abre la URL impresa e inicia sesión con tu cuenta `cientifico`. Confirma el código.
**Esperado:** los eventos previos se ligan a tu usuario con `ATRIBUCION_SESION`, sin reescribir la bitácora.

In [ ]:
resultado = audit_tracer.login(api_url=API_URL)
from audit_tracer.device_login import load_cached_identity
identidad = load_cached_identity()
TOKEN, USUARIO = identidad["token"], identidad["usuario_id"]
check("A2", "Login por dispositivo exitoso", resultado.get("success"), USUARIO)
atribucion = ultimos_eventos(tipo_accion="ATRIBUCION_SESION")
check("A2", "Evento ATRIBUCION_SESION registrado", bool(atribucion), atribucion[0]["contexto_ejecucion"] if atribucion else "ninguno")
original = ultimos_eventos(tipo_accion="CARGA", usuario_id="DESCONOCIDO")
check("A2", "El evento original sigue intacto como DESCONOCIDO", bool(original))

## A3 · Análisis clínico sin cambios de código
**Propiedad:** captura no invasiva.
**Esperado:** se registra cada operación y el archivo resultante es idéntico al de una ejecución sin Audit Tracer.

In [ ]:
ANALISIS = '''
df = pd.read_csv("pacientes_sinteticos.csv")
_ = df["edad"]
adultos = df.query("edad >= 18")
limpio = adultos.fillna({"glucosa": 100})
limpio = limpio.dropna(subset=["sexo"])
limpio = limpio.rename(columns={"presion": "presion_sistolica"})
limpio = limpio.drop(columns=["notas"])
limpio = limpio.assign(glucosa_mmol=limpio["glucosa"].apply(lambda g: round(g / 18, 2)))
unido = limpio.merge(pd.read_csv("diagnosticos.csv"), on="codigo_dx")
unido.head(200).to_csv(SALIDA, index=False)
'''

desde = ultimos_eventos()[0]["event_id"]
inicio = time.perf_counter()
exec(ANALISIS, {"pd": pd, "SALIDA": "salida_auditada.csv"}, globals())
print(f"Análisis auditado: {time.perf_counter() - inicio:.2f} s")

control = "import pandas as pd\nSALIDA='salida_control.csv'\n" + ANALISIS
subprocess.check_call([sys.executable, "-c", control])

conn = sqlite3.connect(LOCAL)
tipos = dict(conn.execute("SELECT tipo_accion, COUNT(*) FROM audit_log WHERE event_id > ? GROUP BY 1", (desde,)).fetchall())
firmados = conn.execute("SELECT COUNT(*) FROM audit_log WHERE event_id > ? AND firma_digital IS NOT NULL", (desde,)).fetchone()[0]
total = conn.execute("SELECT COUNT(*) FROM audit_log WHERE event_id > ?", (desde,)).fetchone()[0]
conn.close()
print(tipos)
check("A3", "Se registraron carga, consulta, transformación y exportación",
      all(tipos.get(t) for t in ("CARGA", "CONSULTA", "TRANSFORMACION", "EXPORTACION")), tipos)
check("A3", "Todos los eventos quedaron firmados", total and firmados == total, f"{firmados}/{total}")
check("A3", "La salida es idéntica a la de una ejecución sin auditoría",
      pathlib.Path("salida_auditada.csv").read_bytes() == pathlib.Path("salida_control.csv").read_bytes())

## A4 · Los errores del usuario llegan intactos
**Esperado:** el mismo `FileNotFoundError` de pandas y un evento `ACCESO_FALLIDO`.

In [ ]:
capturado = None
try:
    pd.read_csv("no_existe.csv")
except FileNotFoundError as exc:
    capturado = exc
check("A4", "La excepción original llega al usuario", capturado is not None, type(capturado).__name__)
e = ultimos_eventos()[0]
check("A4", "El intento queda como ACCESO_FALLIDO", e["tipo_accion"] == "ACCESO_FALLIDO", e["motivo_fallo"])

## A5 · Exportación masiva
**Propiedad:** detección de posible exfiltración.
**Esperado:** una exportación de más de 1.000 filas queda en `CRITICO`.

In [ ]:
unido.to_csv("extraccion_masiva.csv", index=False)
EXPORTACION = ultimos_eventos(tipo_accion="EXPORTACION")[0]
check("A5", "Exportación masiva clasificada como CRITICO",
      EXPORTACION["filas_exportadas"] > 1000 and EXPORTACION["nivel_alerta"] == "CRITICO",
      f"{EXPORTACION['filas_exportadas']} filas · {EXPORTACION['nivel_alerta']} · {EXPORTACION['motivo_alerta']}")


def esperar_sincronizacion(segundos=90):
    fin = time.time() + segundos
    while time.time() < fin:
        audit_tracer.sync_now()
        estado = audit_tracer.sync_status()
        if not estado["pendientes"] and not estado["fallidos"]:
            return estado
        time.sleep(3)
    return estado


estado = esperar_sincronizacion()
check("A5", "Todos los eventos llegaron al servidor central", not estado["pendientes"] and not estado["fallidos"], estado)

## A6 · Servidor central caído
**Propiedad:** resiliencia; la auditoría no frena al usuario y no pierde eventos.
**Esperado:** los eventos quedan en cola local y se envían al volver la conexión.

In [ ]:
from audit_tracer import sync_client

sync_client.configure_sync("http://127.0.0.1:9", TOKEN, auto=False)   # puerto sin servicio
inicio = time.perf_counter()
unido.head(10).to_csv("sin_conexion.csv", index=False)
demora = time.perf_counter() - inicio
EVENTO_OFFLINE = ultimos_eventos(tipo_accion="EXPORTACION")[0]
audit_tracer.sync_now()
estado = audit_tracer.sync_status()
check("A6", "La operación no espera al servidor", demora < 3, f"{demora:.2f} s")
check("A6", "El evento queda guardado en la cola local", estado["pendientes"] + estado["fallidos"] >= 1, estado)

sync_client.configure_sync(API_URL, TOKEN, auto=False)
estado = esperar_sincronizacion()
check("A6", "Al volver la conexión, la cola se vacía", not estado["pendientes"] and not estado["fallidos"], estado)

---
# Bloque B · Integridad y no repudio

Aquí actúas como un atacante con acceso al archivo SQLite local.

## B1 · UPDATE y DELETE directos
**Esperado:** los triggers rechazan la operación y dejan constancia en `audit_intentos_bloqueados`.

In [ ]:
atacante = sqlite3.connect(LOCAL)
objetivo = EXPORTACION["event_id"]
previos = atacante.execute("SELECT COUNT(*) FROM audit_intentos_bloqueados").fetchone()[0]

for sentencia, nombre in [(f"UPDATE audit_log SET filas_exportadas = 1 WHERE event_id = {objetivo}", "UPDATE"),
                          (f"DELETE FROM audit_log WHERE event_id = {objetivo}", "DELETE")]:
    error = None
    try:
        atacante.execute(sentencia)
    except sqlite3.DatabaseError as exc:
        error = str(exc)
    atacante.commit()
    check("B1", f"{nombre} directo rechazado", error is not None, error)

filas = atacante.execute("SELECT filas_exportadas FROM audit_log WHERE event_id = ?", (objetivo,)).fetchone()
intentos = atacante.execute("SELECT COUNT(*) FROM audit_intentos_bloqueados").fetchone()[0] - previos
atacante.close()
check("B1", "El registro sigue intacto", filas and filas[0] == EXPORTACION["filas_exportadas"], filas)
check("B1", "Los dos intentos quedaron registrados", intentos == 2, intentos)

## B2 · Borrar los triggers y alterar el registro
Se trabaja sobre una copia de la base para no afectar las pruebas siguientes.
**Esperado:** `verify_integrity()` señala solo el registro alterado.

## B3 · Alterar y recalcular el SHA-256
**Esperado:** el hash ya no lo detecta (límite conocido), pero la firma Ed25519 sí.

In [ ]:
from audit_tracer.models.audit_log import verify_integrity, _HASHED_COLUMNS
from audit_tracer.digital_signatures import (get_or_create_user_keypair, public_key_to_pem, verify_signature,
                                             sign_event, generate_keypair, _SIGNED_FIELDS)
from audit_tracer.utils.hashing import hash_event

LLAVE_PRIVADA, llave_publica = get_or_create_user_keypair()
PEM = public_key_to_pem(llave_publica)


def copia_atacante():
    ruta = str(TRABAJO / f"copia_{uuid.uuid4().hex[:6]}.db")
    origen, destino = sqlite3.connect(LOCAL), sqlite3.connect(ruta)
    origen.backup(destino)
    origen.close()
    destino.execute("DROP TRIGGER IF EXISTS trg_audit_log_no_update")
    destino.execute("DROP TRIGGER IF EXISTS trg_audit_log_no_delete")
    destino.commit()
    return destino


# B2: alteración simple
copia = copia_atacante()
copia.execute("UPDATE audit_log SET filas_exportadas = 1 WHERE event_id = ?", (objetivo,))
copia.commit()
corruptos = verify_integrity(copia)
check("B2", "Solo el registro alterado aparece como corrupto",
      [c["event_id"] for c in corruptos] == [objetivo], [c["event_id"] for c in corruptos])

# B3: alteración con hash recalculado
fila = dict(copia.execute("SELECT * FROM audit_log WHERE event_id = ?", (objetivo,)).fetchone())
nuevo_hash = hash_event({c: fila[c] for c in _HASHED_COLUMNS})
copia.execute("UPDATE audit_log SET hash_integridad = ? WHERE event_id = ?", (nuevo_hash, objetivo))
copia.commit()
check("B3", "Límite conocido: con el hash recalculado, verify_integrity no lo detecta", verify_integrity(copia) == [])
firma = verify_signature(copia, objetivo, PEM)
check("B3", "La firma Ed25519 detecta la alteración", firma["firma_presente"] and not firma["firma_valida"], firma["mensaje"])

## B4 · Atacante con la llave privada del cliente
La llave vive en el equipo del usuario. Con ella, un atacante puede volver a firmar el registro alterado.
**Esperado:** la copia local pasa la verificación (límite conocido). El servidor conserva el evento original.

In [ ]:
fila = dict(copia.execute("SELECT * FROM audit_log WHERE event_id = ?", (objetivo,)).fetchone())
firma_nueva = sign_event(fila, LLAVE_PRIVADA)
copia.execute("UPDATE audit_log SET firma_digital = ? WHERE event_id = ?", (firma_nueva, objetivo))
copia.commit()
check("B4", "Límite conocido: la firma rehecha pasa en la copia local", verify_signature(copia, objetivo, PEM)["firma_valida"])
copia.close()

alterado = {c: fila[c] for c in _SIGNED_FIELDS}
alterado["firma_digital"] = firma_nueva
r = sincronizar(TOKEN, alterado)
ID_SERVIDOR = r.json().get("event_id")
check("B4", "El servidor no sobrescribe: responde duplicado", r.status_code == 200 and r.json().get("status") == "duplicado", r.json())

auditor = sesion_web("auditor")
prueba = auditor.get(f"{API_URL}/merkle/verificar/{ID_SERVIDOR}", timeout=TIMEOUT).json()
check("B4", "La copia central conserva el hash original",
      prueba.get("leaf_hash") == EXPORTACION["hash_integridad"] != nuevo_hash, str(prueba.get("leaf_hash"))[:16])

## B5 · Eventos manipulados en tránsito
**Propiedad:** el servidor solo acepta eventos con firma válida del dueño del token.
**Esperado:** HTTP 422 en cada manipulación, 201 en el control y 200 `duplicado` en el reenvío.

In [ ]:
def evento_firmado(llave=None, **extra):
    evento = {"evento_uuid": str(uuid.uuid4()), "usuario_id": USUARIO, "sesion_id": "lab-b5",
              "timestamp": dt.datetime.now(dt.timezone.utc).replace(tzinfo=None).isoformat(), "tipo_accion": "EXPORTACION",
              "dataset_nombre": "laboratorio.csv", "ruta_destino": "/content/lab.csv",
              "filas_exportadas": 10, "nivel_alerta": "NORMAL", **extra}
    evento["firma_digital"] = sign_event(evento, llave or LLAVE_PRIVADA)
    return evento


control = evento_firmado()
r = sincronizar(TOKEN, control)
check("B5", "Control: evento íntegro aceptado", r.status_code == 201, r.status_code)
r = sincronizar(TOKEN, control)
check("B5", "Reenvío (replay) sin duplicar", r.status_code == 200 and r.json().get("status") == "duplicado", r.json().get("status"))

for campo, valor in [("filas_exportadas", 1), ("nivel_alerta", "CRITICO"), ("dataset_nombre", "otro.csv"),
                     ("timestamp", "2020-01-01T00:00:00"), ("ruta_destino", "/tmp/x.csv")]:
    r = sincronizar(TOKEN, {**evento_firmado(), campo: valor})
    check("B5", f"Campo {campo} alterado tras firmar", r.status_code == 422, r.status_code)

r = sincronizar(TOKEN, evento_firmado(llave=generate_keypair()[0]))
check("B5", "Firma hecha con la llave de un atacante", r.status_code == 422, r.status_code)
r = sincronizar(TOKEN, {**evento_firmado(), "firma_digital": "AAAA"})
check("B5", "Firma basura", r.status_code == 422, r.status_code)
sin_firma = evento_firmado()
sin_firma.pop("firma_digital")
r = sincronizar(TOKEN, sin_firma)
check("B5", "Degradación: evento sin firma", r.status_code == 422, r.status_code)
check("B5", "Los rechazos quedan como FIRMA_INVALIDA", filas_eventos(auditor, tipo_accion="FIRMA_INVALIDA") > 0)

## B6 · Árbol de Merkle y ancla externa
Se usa una base temporal para poder alterar un lote ya anclado. El código es el mismo que corre en el servidor.
**Esperado:** la prueba de inclusión falla con una hoja alterada, y el ancla detecta la modificación del lote.

In [ ]:
from audit_tracer.db import get_connection
from audit_tracer.models.audit_log import insert_event
from audit_tracer.merkle import generate_proof_for_event, verify_merkle_proof, publish_batch_root, verify_external_anchor

ruta_lote = str(TRABAJO / "lote_merkle.db")
anclas = str(TRABAJO / "anclas")
HORA = "2026-11-09T10"
lote = get_connection(ruta_lote)
ids = [insert_event(lote, {"usuario_id": "u-lab", "sesion_id": "merkle", "timestamp": f"{HORA}:{i:02d}:00",
                           "tipo_accion": "CONSULTA", "dataset_nombre": "lote.csv", "nivel_alerta": "NORMAL"})
       for i in range(16)]

prueba = generate_proof_for_event(lote, ids[6])
check("B6", "La prueba de inclusión es válida", verify_merkle_proof(prueba), f"{len(prueba['proof'])} pasos para 16 eventos")
check("B6", "Una hoja alterada invalida la prueba", not verify_merkle_proof({**prueba, "leaf_hash": "0" * 64}))

publish_batch_root(lote, HORA, anchors_dir=anclas)
antes = verify_external_anchor(lote, HORA, anchors_dir=anclas)
check("B6", "El ancla firmada coincide antes de alterar", antes["integrity_ok"] and antes["signature_valid"], antes["details"])
lote.close()

crudo = sqlite3.connect(ruta_lote)
crudo.execute("DROP TRIGGER IF EXISTS trg_audit_log_no_update")
crudo.execute("UPDATE audit_log SET hash_integridad = ? WHERE event_id = ?", ("f" * 64, ids[3]))
crudo.commit()
despues = verify_external_anchor(crudo, HORA, anchors_dir=anclas)
crudo.close()
check("B6", "El ancla detecta el lote alterado", not despues["integrity_ok"], despues["details"])

---
# Bloque C · Autenticación, tokens y control de acceso

## C1 · Dos secretos en el flujo de dispositivo
**Propiedad:** ver el código corto no basta para obtener el token, y el token se entrega una sola vez.
**Qué hacer:** abre la URL impresa con tu cuenta `cientifico` y confirma. Este token se usa también en C3.

In [ ]:
inicio = requests.post(f"{API_URL}/api/auth/dispositivo/iniciar", timeout=TIMEOUT).json()
print("Abre y confirma:", inicio["url_activacion"])
print("Código:", inicio["codigo"])


def consultar(device_code):
    return requests.get(f"{API_URL}/api/auth/dispositivo/estado", params={"device_code": device_code}, timeout=TIMEOUT)


r = consultar(inicio["codigo"])
check("C1", "El código corto no sirve como credencial", r.status_code == 404, r.status_code)

fin = time.time() + 600
while time.time() < fin:
    r = consultar(inicio["device_code"])
    if r.json().get("estado") != "PENDIENTE":
        break
    time.sleep(3)
TOKEN_DESECHABLE = r.json().get("token")
check("C1", "El token se entrega al confirmar", bool(TOKEN_DESECHABLE), r.json().get("estado"))
r = consultar(inicio["device_code"])
check("C1", "El token no se entrega dos veces", r.status_code == 410, r.status_code)

## C2 · Tokens inválidos y HTTPS obligatorio
**Esperado:** HTTP 401 con tokens alterados, inventados o ausentes. El cliente rechaza URLs `http://` externas.

In [ ]:
casos = {"alterado en un carácter": TOKEN[:-1] + ("a" if TOKEN[-1] != "a" else "b"),
         "inventado": uuid.uuid4().hex, "ausente": None}
for nombre, valor in casos.items():
    r = sincronizar(valor, evento_firmado())
    check("C2", f"Token {nombre}", r.status_code == 401, r.status_code)

error = None
try:
    sync_client.configure_sync("http://servidor.example.org", TOKEN, auto=False)
except ValueError as exc:
    error = str(exc)
check("C2", "El cliente rechaza HTTP sin TLS", error is not None, error)

## C3 · Revocación inmediata
**Esperado:** el mismo token funciona (201) y, una vez revocado, recibe 401.

In [ ]:
admin = sesion_web("admin")
r = sincronizar(TOKEN_DESECHABLE, evento_firmado())
check("C3", "El token funciona antes de revocarlo", r.status_code == 201, r.status_code)
r = admin.post(f"{API_URL}/api/tokens/revocar", json={"token": TOKEN_DESECHABLE, "motivo": "Prueba C3"}, timeout=TIMEOUT)
check("C3", "El administrador lo revoca", r.status_code == 200, r.json().get("message"))
r = sincronizar(TOKEN_DESECHABLE, evento_firmado())
check("C3", "El token revocado se rechaza", r.status_code == 401, r.json().get("mensaje"))

## C4 · Control de acceso por rol
**Propiedad:** mínimo privilegio y rastro de cada intento.
**Esperado:** ninguna escalada funciona y cada intento queda como `ACCESO_DENEGADO`.

In [ ]:
cientifico, analista = sesion_web("cientifico"), sesion_web("analista")

r = cientifico.get(f"{API_URL}/eventos", allow_redirects=False, timeout=TIMEOUT)
check("C4", "CIENTIFICO_DATOS no consulta eventos", r.status_code == 403, r.status_code)
r = cientifico.post(f"{API_URL}/merkle/publicar", json={}, allow_redirects=False, timeout=TIMEOUT)
check("C4", "CIENTIFICO_DATOS no publica anclas", r.status_code == 403, r.status_code)

correo_nuevo = f"escalada.{uuid.uuid4().hex[:6]}@validacion.audit"
r = analista.post(f"{API_URL}/admin/usuarios/nuevo", allow_redirects=False, timeout=TIMEOUT,
                  data={"email": correo_nuevo, "password": "Escalada123*", "confirmar_password": "Escalada123*", "rol": "ADMIN"})
check("C4", "ANALISTA no puede crear un ADMIN", r.status_code == 302 and "/dashboard" in r.headers.get("Location", ""), r.headers.get("Location"))
existe = correo_nuevo in admin.get(f"{API_URL}/admin/usuarios", timeout=TIMEOUT).text
check("C4", "La cuenta no se creó", not existe)
r = analista.post(f"{API_URL}/admin/tokens/nuevo", data={"usuario_id": admin.usuario_id}, allow_redirects=False, timeout=TIMEOUT)
check("C4", "ANALISTA no emite tokens ajenos", r.status_code == 302 and "/dashboard" in r.headers.get("Location", ""), r.headers.get("Location"))

for nombre, sesion in (("CIENTIFICO_DATOS", cientifico), ("ANALISTA", analista)):
    check("C4", f"Intentos de {nombre} visibles para el auditor",
          filas_eventos(auditor, tipo_accion="ACCESO_DENEGADO", usuario_id=sesion.usuario_id) >= 1)

## C5 · Cookie de sesión forjada
**Propiedad:** no se puede escalar privilegios falsificando la cookie.
**Esperado:** una cookie con `rol=ADMIN` firmada con una clave incorrecta se trata como sesión inexistente.

In [ ]:
from flask import Flask
from flask.sessions import SecureCookieSessionInterface


def forjar(clave):
    app = Flask("forja")
    app.secret_key = clave
    firmador = SecureCookieSessionInterface().get_signing_serializer(app)
    return firmador.dumps({"usuario_id": "atacante", "rol": "ADMIN", "nombre": "atacante", "token_id": "x"})


for nombre, clave in [("clave antigua publicada en el repositorio", "audit_tracer_dev_key_2026"), ("clave aleatoria", uuid.uuid4().hex)]:
    r = requests.get(f"{API_URL}/admin/usuarios", cookies={"session": forjar(clave)}, allow_redirects=False, timeout=TIMEOUT)
    check("C5", f"Cookie forjada con {nombre}", r.status_code == 302 and "/login" in r.headers.get("Location", ""), r.status_code)

## C6 · Fuerza bruta
**Propiedad:** bloqueo automático de la cuenta.
**Esperado:** tras 5 contraseñas erróneas, la cuenta `bloqueo` no entra ni con la contraseña correcta.

In [ ]:
def intentar(clave):
    return requests.post(f"{API_URL}/login", data={"email": correo("bloqueo"), "password": clave},
                         allow_redirects=False, timeout=TIMEOUT)


for i in range(5):
    r = intentar(f"incorrecta-{i}")
check("C6", "La quinta falla bloquea la cuenta", "bloqueada" in r.text.lower(), r.status_code)
r = intentar(contrasena("bloqueo"))
check("C6", "La contraseña correcta ya no entra", r.status_code != 302 and "bloqueada" in r.text.lower(), r.status_code)

## C7 · Reutilizar la cookie después de cerrar sesión
**Propiedad:** el cierre de sesión invalida la sesión en el servidor, no solo en el navegador.
**Esperado:** la cookie copiada antes del cierre deja de servir.

In [ ]:
victima = sesion_web("cientifico")
robada = victima.cookies["session"]
r = requests.get(f"{API_URL}/dashboard", cookies={"session": robada}, allow_redirects=False, timeout=TIMEOUT)
check("C7", "La cookie funciona antes de cerrar sesión", r.status_code == 200, r.status_code)
victima.get(f"{API_URL}/logout", timeout=TIMEOUT)
r = requests.get(f"{API_URL}/dashboard", cookies={"session": robada}, allow_redirects=False, timeout=TIMEOUT)
check("C7", "La cookie copiada se rechaza tras el cierre", r.status_code == 302 and "/login" in r.headers.get("Location", ""), r.status_code)

---
# Resumen
Copia este resultado en el registro de hallazgos de la guía. Luego continúa con el **Bloque D** en el dashboard.

In [ ]:
audit_tracer.sync_now()
resumen = pd.DataFrame(RESULTADOS)
por_escenario = resumen.groupby("escenario")["cumple"].agg(["sum", "count"]).rename(columns={"sum": "cumplen", "count": "pruebas"})
print(por_escenario.to_string())
print(f"\nTotal: {int(resumen['cumple'].sum())} de {len(resumen)} pruebas cumplen.")
fallas = resumen[~resumen["cumple"]]
if len(fallas):
    print("\nNo cumplen:")
    print(fallas[["escenario", "prueba", "obtenido"]].to_string(index=False))